# Parts 3–4 · 04 Sample

The event-level table every other part of the project uses. It:

- joins surprise (02) and returns (03);
- links each event to its 8-K in Mahaut's filing sample, giving the accession number the text is keyed on;
- adds every sample rule as a **flag, not a deletion**, so a rule can change without rebuilding anything;
- writes the final funnel and the hand-off files.

**Flags:** `f_analysts3`, `f_price5`, `f_size` (NYSE 20th percentile, not cut on), `two_source` (wire and I/B/E/S agree), `timing_suspect` (I/B/E/S-only intraday times), `has_filing`, `basis_suspect` (I/B/E/S actual looks like GAAP while analysts forecast operating EPS).

**Samples:**
- `in_primary`: 3+ analysts, price above $5, a filing, two sources agreeing, a reaction return. This is the pre-registered primary sample.
- `in_text_sample`: the same without the timing requirement, for robustness.

**Needs:** `common.py`, `returns.py`, `sample.py` in `code/`; `data/surprise.parquet`, `data/returns.parquet`, `data/clock_provisional.csv`; `cache/`. There's no WRDS connection.

**Writes:**
- `data/events_final.parquet` (all events, all flags; WRDS-derived, stays private);
- `data/text_sample_keys.csv` (which filings to score: identifiers and dates only);
- `results/funnel_final.csv`, `results/sample_by_split.csv`, `results/gaap_basis_check.csv`, `results/robustness_ic_dev.csv`.

In [4]:
import os, sys
from pathlib import Path

# Find the checkout from the repository root or any notebook subfolder.
_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts").is_dir()
                  and (p / "3_4_surprise_returns" / "code").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned S08 repository, "
                            "or set S08_ROOT to its path.")
# Optional private data root; code always comes from this checkout.
TEAM_DIR = Path(os.environ.get("TEAM_DIR", REPO_ROOT)).expanduser().resolve()
os.environ["TEAM_DIR"] = str(TEAM_DIR)
SCRIPTS = REPO_ROOT / "scripts"
NOV2_DATA = TEAM_DIR / "1_2nov" / "nov2_data"
if not NOV2_DATA.is_dir():
    NOV2_DATA = TEAM_DIR / "files csv"


PROJECT_DIR = TEAM_DIR / "3_4_surprise_returns"
os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
sys.path.insert(0, str(REPO_ROOT / "3_4_surprise_returns" / "code"))

need = [REPO_ROOT / "3_4_surprise_returns" / "code" / f for f in ["common.py", "surprise.py", "returns.py", "sample.py"]]
need += [PROJECT_DIR / "data" / f for f in ["surprise.parquet", "returns.parquet", "clock_provisional.csv"]]
missing = [str(p) for p in need if not p.exists()]
assert not missing, f"missing: {missing}"
print("project folder:", PROJECT_DIR)

Mounted at /content/drive
project folder: /content/drive/MyDrive/S08 NLP/3_4_surprise_returns


## Load, and check that 02 and 03 line up

In [5]:
import importlib, time
import numpy as np
import pandas as pd
import common, surprise as S, returns as Rm, sample as SM
for m in (common, S, Rm, SM):
    importlib.reload(m)

t0 = time.time()
s = pd.read_parquet(PROJECT_DIR / "data" / "surprise.parquet")
ret = pd.read_parquet(PROJECT_DIR / "data" / "returns.parquet")
clk = pd.read_csv(PROJECT_DIR / "data" / "clock_provisional.csv",
                  usecols=["accession", "permno", "cik", "filing_date", "reaction_date"],
                  dtype={"accession": str, "cik": str})
cap = S.load_cache("crsp_daily", columns=["permno", "date", "dlycap"])
cap = cap[cap["permno"].isin(s["permno"].unique())]
bp = S.load_cache("nyse_breakpoints")
ccm = S.load_cache("link_ccm")
fq = S.load_cache("comp_fundq", columns=["gvkey", "datadate", "epsfxq", "oepsxq"])

assert len(s) == len(ret) and set(s["event_id"]) == set(ret["event_id"]), "02 and 03 are out of step: rerun 03"
print(f"events {len(s):,} | versions: {s['surprise_version'].iloc[0]}, {ret['returns_version'].iloc[0]} | "
      f"filings in Mahaut's sample {clk['accession'].nunique():,} | {time.time() - t0:.0f}s")

events 217,611 | versions: surprise_v2_wire_first, returns_v3_placebo_day_m1 | filings in Mahaut's sample 115,976 | 12s


## Build

In [6]:
links = SM.link_filings(s, clk)
size = SM.add_size(s, cap, bp)
gaap = SM.add_gaap(s, ccm, fq)
e = SM.build_final(s, ret, links, size, gaap)

lk = e[e["has_filing"]]
print(f"events linked to an 8-K: {len(lk):,} of {len(e):,} ({len(lk) / len(e):.1%})")
print("filing date minus release date (days):", lk["link_gap_days"].value_counts().sort_index().to_dict())
print("filing's own clock gives the same session:", round(lk["link_same_session"].astype(bool).mean(), 3))

events linked to an 8-K: 104,083 of 217,611 (47.8%)
filing date minus release date (days): {-1.0: 85, 0.0: 93878, 1.0: 8617, 2.0: 682, 3.0: 357, 4.0: 464}
filing's own clock gives the same session: 0.995


## Final funnel and sample sizes

In [7]:
f = SM.funnel(e)
print(f.to_string(index=False))

e["in_primary_large"] = e["in_primary"] & e["f_size"]
tab = SM.by_split(e, ["in_text_sample", "in_primary", "in_primary_large"])
print("\n", tab.to_string())

t = e[e["in_text_sample"]]
print("\ntext sample by timing status (ibes_only rows need EDGAR times to join the primary sample):")
print(pd.crosstab(t["clock_status"], t["split"], margins=True).to_string())
print("\nI/B/E/S-only intraday events in the text sample (timing_suspect):", int(t["timing_suspect"].sum()))

                                                  step  remaining  removed  removed_pct
                   events with SUE and returns (02-03)     217611        0          0.0
                                           3+ analysts     143068    74543         34.3
                                 price above $5 at t-2     131621    11447          8.0
linked to an 8-K in the filing sample (text available)      97787    33834         25.7
   wire and I/B/E/S agree on the session (two sources)      74217    23570         24.1
                             reaction return available      73142     1075          1.4

                     dev  validation  excluded_2020  holdout  after Llama cutoff (2024-25)  total
in_text_sample    51529       14523           5232    26503                          9899  97787
in_primary        35508       11375           4114    22145                          8579  73142
in_primary_large  35131       11304           4096    22060                          8552  

## GAAP versus street: is the I/B/E/S actual on the analysts' basis?

Compares each actual with Compustat's diluted GAAP EPS and diluted operating EPS for the same quarter. `basis_suspect` flags events where the actual equals GAAP, GAAP differs from operating EPS by 5+ cents, and the consensus sits much closer to operating EPS. Those are probable fake surprises. They stay in the data, flagged.

In [8]:
t = e[e["in_text_sample"]]
basis = t["actual_basis"].value_counts()
print("actual_basis, text sample:", basis.to_dict())
print(f"basis_suspect: {int(t['basis_suspect'].sum()):,} of {len(t):,} ({t['basis_suspect'].mean():.2%})")

hc_path = PROJECT_DIR / "data" / "handcheck_sample_50.csv"
if hc_path.exists():
    hc = pd.read_csv(hc_path, dtype=str)
    flagged = e[e["event_id"].isin(hc["event_id"]) & e["basis_suspect"]]
    print("hand-check events flagged as basis_suspect:", flagged["event_id"].tolist())

cols = ["event_id", "cname", "pends", "actual", "gaap_eps", "operating_eps", "consensus_median", "n_analysts"]
print("\nlargest suspects:")
sus = t[t["basis_suspect"]].assign(gap=lambda x: (x["actual"] - x["consensus_median"]).abs())
print(sus.sort_values("gap", ascending=False)[cols].head(10).to_string(index=False))

actual_basis, text sample: {'neither': 53422, 'gaap_and_operating': 25607, 'gaap_only': 9568, 'operating_only': 9144, 'no_compustat': 46}
basis_suspect: 1,131 of 97,787 (1.16%)
hand-check events flagged as basis_suspect: []

largest suspects:
       event_id            cname      pends  actual  gaap_eps  operating_eps  consensus_median  n_analysts
TDOC|2022-12-31   TELADOC HEALTH 2022-12-31  -23.49    -23.49          -0.18             -0.26           5
TDOC|2022-06-30   TELADOC HEALTH 2022-06-30  -19.22    -19.22          -0.42            -0.615          12
CCMM|2016-06-30   CHARTER COMMNS 2016-06-30   15.17     15.17          -0.66             -0.13          11
FABC|2008-12-31 REGIONS FINL COR 2008-12-31   -9.01     -9.01          -0.35             -0.06          17
 KBH|2007-11-30          KB HOME 2007-11-30   -9.99     -9.99          -3.34             -1.34           3
ETSY|2022-09-30             ETSY 2022-09-30   -7.62     -7.62           0.65              0.39           8
 CTX|200

## Robustness: development years only

The SUE-only rank IC in four groups:
- the primary sample;
- its large stocks (at or above the NYSE 20th percentile);
- its small stocks;
- the whole text sample.

Plus the stricter window starting at day 0's close (`car_1_60`), which doesn't depend on the opening price. If the drift survives in large stocks and from the close, it's credible.

In [9]:
d = common.guard_holdout(e)
d = d[d["split"].eq("dev")]
groups = {"primary": d["in_primary"], "primary, large": d["in_primary"] & d["f_size"],
          "primary, small": d["in_primary"] & ~d["f_size"], "text sample": d["in_text_sample"]}
outcomes = ["car_0_1_ff4", "post_car_1_ff4", "post_car_60_ff4", "car_1_60_ff4"]
rows = []
for g, mask in groups.items():
    for o in outcomes:
        r = Rm.rank_ic_by_season(d[mask], "sue_price", o)
        rows.append({"group": g, "n": int(mask.sum()), "outcome": o, "mean_ic": r["mean_ic"], "t_nw": r["t_nw"]})
rob = pd.DataFrame(rows)
print(rob.pivot(index=["group", "n"], columns="outcome", values="mean_ic")[outcomes].round(3).to_string())
print("\nNewey-West t:")
print(rob.pivot(index=["group", "n"], columns="outcome", values="t_nw")[outcomes].round(1).to_string())

holdout guard: dropped 56,056 rows from 2021-2025
outcome               car_0_1_ff4  post_car_1_ff4  post_car_60_ff4  car_1_60_ff4
group          n                                                                
primary        35508        0.352           0.086            0.024         0.002
primary, large 35131        0.352           0.086            0.024         0.002
primary, small 377          0.337           0.079            0.008        -0.052
text sample    51529        0.359           0.091            0.030         0.007

Newey-West t:
outcome               car_0_1_ff4  post_car_1_ff4  post_car_60_ff4  car_1_60_ff4
group          n                                                                
primary        35508         56.1            10.7              2.6           0.2
primary, large 35131         56.2            10.7              2.6           0.2
primary, small 377           13.5             1.9              0.2          -1.2
text sample    51529         51.6           

## Save

In [10]:
(PROJECT_DIR / "results").mkdir(exist_ok=True)
e.to_parquet(PROJECT_DIR / "data" / "events_final.parquet", index=False)
keys = e.loc[e["in_text_sample"], SM.HANDOFF_COLUMNS]
keys.to_csv(PROJECT_DIR / "data" / "text_sample_keys.csv", index=False)
f.to_csv(PROJECT_DIR / "results" / "funnel_final.csv", index=False)
tab.to_csv(PROJECT_DIR / "results" / "sample_by_split.csv")
e[e["in_text_sample"]]["actual_basis"].value_counts().rename("events").to_csv(
    PROJECT_DIR / "results" / "gaap_basis_check.csv")
rob.to_csv(PROJECT_DIR / "results" / "robustness_ic_dev.csv", index=False)
print(f"saved {len(e):,} events; {len(keys):,} filings in text_sample_keys.csv "
      f"({int(keys['in_primary'].sum()):,} in the primary sample)")

saved 217,611 events; 97,787 filings in text_sample_keys.csv (73,142 in the primary sample)


In [11]:
import subprocess
subprocess.run([sys.executable, str(REPO_ROOT / "3_4_surprise_returns" / "code" / "tests" / "run_all.py")], check=True)


== test_surprise.py
look-ahead check passed on 4 estimates
                                                         step  remaining  removed  removed_pct
            I/B/E/S quarterly EPS actuals announced 2006-2025          6        0          0.0
                            one announcement per firm-quarter          5        1         16.7
                                     actual EPS value present          5        0          0.0
                                actual reported in US dollars          4        1         20.0
linked to a CRSP permno on the announcement date (score <= 3)          3        1         25.0
                            announcement time known (I/B/E/S)          2        1         33.3
           at least one point-in-time estimate within 90 days          2        0          0.0
                           stock is a US common share in CRSP          2        0          0.0
                                            CRSP price at t-2          2        0     